# 06 · Six volume features and the trade-day opening gap

Combine the completed **eight momentum inputs + six volume inputs + one opening
gap = 15 inputs**. The first fourteen are known at the previous completed regular
close. The gap is morning information on the day of the trade.

This notebook builds/loads features, audits availability and prepares XGBoost
matrices. It does **not fit a new model**. It uses existing five-minute data only;
the user deferred one-minute and delayed-execution analysis. All data and notebook
outputs remain private under Git-ignored `data/`.

**Workspace review:** source notebooks contain no private outputs. Model training and feature rebuilding are disabled by default. Read `docs/AUDIT_FINDINGS.md` before interpreting results.

In [ ]:
from pathlib import Path
import json, os, sys
PROJECT_DIR=next(p for p in [Path.cwd(),*Path.cwd().parents]
    if (p / "scripts/build_volume_overnight_features.py").is_file())
sys.path.insert(0,str(PROJECT_DIR / "scripts"))
from build_volume_overnight_features import build,FEATURE_SETS,MODEL_FEATURES
from volume_features import VOLUME_FEATURES
FEATURE_DIR=PROJECT_DIR / "data/features/06_momentum_volume_overnight_v1"
REPORT_DIR=PROJECT_DIR / "data/reports/06_build_volume_and_overnight_features"
REPORT_DIR.mkdir(parents=True,exist_ok=True)
if not (FEATURE_DIR / "MANIFEST.json").exists():
    raise FileNotFoundError("Pinned feature release missing; inspection does not build it automatically.")
import duckdb
import numpy as np
import pandas as pd
import xgboost as xgb
from IPython.display import display,Markdown
pd.set_option("display.max_columns",24)
manifest=json.loads((FEATURE_DIR / "MANIFEST.json").read_text())
audit=json.loads((FEATURE_DIR / "AUDIT.json").read_text())
assert len(MODEL_FEATURES)==15 and len(VOLUME_FEATURES)==6
assert len(FEATURE_SETS["momentum_volume14"])==14
assert manifest["summary"]["forecast_cohort_identical"]
assert manifest["summary"]["original_eight_inputs_labels_outcomes_identical"]
display(pd.DataFrame.from_dict(manifest["summary"],orient="index",columns=["release_summary"]))

## Six volume definitions

Dollar volume is a proxy computed as **raw bar VWAP × raw shares**, summed across
the actual regular session. Price and volume use the same raw adjustment. Session
length and closing-hour windows follow the XNYS calendar, including early closes.
All references exclude the current observation. Signed activity uses the bar's
close-minus-open direction and is not observed buyer/seller-initiated flow.

In [ ]:
definitions=pd.DataFrame([{"feature":k,"definition":v,"available":"history-session close"}
    for k,v in manifest["volume_definitions"].items()])
display(definitions)
availability=pd.read_parquet(FEATURE_DIR / "feature_availability.parquet")
availability["missing_pct"]=100*availability["missing"]/availability.n_rows
display(availability[["feature","n_rows","missing","missing_pct","infinite","minimum","maximum"]])
assert availability.infinite.eq(0).all()
display(pd.read_parquet(FEATURE_DIR / "opening_gap_status_counts.parquet"))

## Information timing and the optimistic opening benchmark

The gap is `today_adjusted_open / previous_adjusted_close - 1`. It is missing
when an opening record is unavailable/duplicated/invalid or its adjustment-source
tag differs from the previous close. These missing inputs do not remove examples.
No missing volume or gap is filled with zero; XGBoost's native NaN handling is
used when matrices are prepared below.

The 15-input decision event is today's opening; the history cutoff remains
yesterday's close. The stored opening five-minute bar normally becomes available
five minutes later. Using its opening field as instantly observed information,
then entering at that same price, is an **optimistic research assumption**.
No opening-bar close/high/low/volume or later trade-day source field enters the
opening feature. One-minute execution analysis is deferred.

In [ ]:
con=duckdb.connect()
con.execute("SET threads=2")
path=FEATURE_DIR / "supervised_panel.parquet"
sample=con.execute("SELECT security_id,ticker,signal_date,trade_date,history_cutoff_utc,"
    "signal_cutoff_utc,opening_bar_available_at_utc,"+",".join(VOLUME_FEATURES)+
    ",overnight_gap,opening_gap_status FROM read_parquet(?) "
    "WHERE opening_gap_status='observed' ORDER BY trade_date DESC,security_id LIMIT 8",[str(path)]).df()
display(sample)
sample.to_parquet(REPORT_DIR / "sample_features.parquet",index=False)
print("No one-minute data were read or downloaded.")

## Preserve the existing experiment

Original history eligibility, forecast identities, eight momentum values, class
labels and observed/missing outcomes are preserved. The 14-input candidate uses
`history_cutoff_utc`; the 15-input candidate uses the optimistic opening cutoff.
Only the added opening feature uses trade-day morning information. Labels remain
that day's open-to-close return deciles, with class 1 highest and class 10 lowest.

The following cell prepares the first 12-month/3-month fold's matrices. It keeps
all forecast rows, including unknown future outcomes. It trains **no model**.

In [ ]:
from ret_backtest import folds
fold=folds()[0]
frame=con.execute("SELECT * FROM read_parquet(?) WHERE trade_date>=CAST(? AS DATE) "
    "AND trade_date<CAST(? AS DATE) ORDER BY trade_date,security_id",
    [str(path),fold["train_start"],fold["test_end"]]).df()
train=frame[frame.trade_date.lt(pd.Timestamp(fold["test_start"])) & frame.realized_class.notna()]
test=frame[frame.trade_date.ge(pd.Timestamp(fold["test_start"]))]
assert train.trade_date.lt(pd.Timestamp(fold["test_start"])).all()
matrices={}
matrix_summary=[]
for name in ["momentum_volume14","momentum_volume_gap15"]:
    cols=FEATURE_SETS[name]
    x_train=train[cols].to_numpy(dtype=np.float32)
    x_test=test[cols].to_numpy(dtype=np.float32)
    assert not np.isinf(x_train).any() and not np.isinf(x_test).any()
    matrices[name]=(xgb.DMatrix(x_train,label=train.realized_class.to_numpy(dtype=np.float32)-1,
        feature_names=cols,missing=np.nan,nthread=1),
        xgb.DMatrix(x_test,feature_names=cols,missing=np.nan,nthread=1))
    matrix_summary.append(dict(feature_set=name,inputs=len(cols),training_rows=len(train),
        forecast_rows=len(test),missing_training_values=int(np.isnan(x_train).sum()),
        missing_forecast_values=int(np.isnan(x_test).sum()),
        cutoff="previous close" if len(cols)==14 else "opening event, optimistic"))
display(pd.DataFrame(matrix_summary))
print("XGBoost matrices prepared; no fit or portfolio result is produced by this notebook.")
con.close()
(REPORT_DIR / "SUMMARY.json").write_text(json.dumps(dict(status="complete",
    feature_release_summary=manifest["summary"],matrices=matrix_summary,
    availability=availability.to_dict("records"),new_model_fitted=False,
    one_minute_data_used=False),indent=2,default=str)+"\n")

For subsequent fitting, retain the selected rolling 12/3-month WFO, 100
randomized fits per quarter, N=2/3 equal-weight legs and current tree settings.
Use validation stopping after 30 non-improving rounds, with no arbitrary boosting
maximum, fixed training RAM ceiling or default time deadline. Compare the volume
addition and opening-gap addition separately when attributing performance.

See `docs/07_VOLUME_AND_OPENING_GAP.md` for formulas and missing-value policy.
The inherited momentum clipping remains unchanged and effectively nonbinding;
no new feature or realized-return clipping is added here.